# 📋 [OPS08_M08] AAR 회의를 통한 보고 지연 및 사격 구역 중복 문제점 분석
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북 (학생 실습용 워크북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/ops08_m08_student.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (Level 2)
- **미션 번호:** OPS08_M08
- **미션 주제:** AAR 회의를 통한 보고 지연 및 사격 구역 중복 문제점 분석
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고 교관 지휘에 엄격히 따른다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 08. AAR 회의를 통한 보고 지연 및 사격 구역 중복 문제점 분석

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"참여자 진술과 실제 관측 로그(F1~F7)를 팩트체크하고, 4M(절차/도구/숙련/환경) 원인 분석을 통해 비난 없는 SOP 1차 개선점 5건을 도출해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 주관적 비난 표현이 배제되었는지, 현행-수정-근거 3단 표가 실제 훈련 결함 데이터에 부합하는지 검증합니다.

---

### 📑 실습자료 및 과업 지침

AAR 회의를 통한 보고 지연 및 사격 구역 중복 문제점 분석**

가. 제공 자료 - AAR 4단계 진행 순서(권장 60분)

| **단계** | **질문** | **시간** |
| --- | --- | --- |
| **1** | 무엇을 하려고 했나(계획·SOP 기준)? | 10분 |
| **2** | 실제로 무슨 일이 있었나(기록 기준)? | 15분 |
| **3** | 왜 차이가 났나(원인)? | 20분 |
| **4** | 다음에는 무엇을 바꿀까(개선)? | 15분 |

나. 제공 자료 - 가상 기록 발췌(팩트체크 연습용) [교육용가상자료]

| **번호** | **출처** | **내용** |
| --- | --- | --- |
| **F1** | 관측자 진술 | "드론을 보자마자 30초 안에 보고했다." |
| **F2** | 시각 기록표 | 최초 포착 05:43:05, 지휘자 수신 05:44:55 (110초) |
| **F3** | 통신 기록 | 05:43:20~05:44:40 사이 지휘자 무전이 다른 보고로 점유됨 |
| **F4** | 구역 할당표 | 1분대 11시~3시, 2분대 2시~6시 (2~3시 중복) |
| **F5** | 평가관 기록 | 2시 표적에 1분대·2분대가 동시에 '사격 준비' 구두 선언 |
| **F6** | 지휘자 진술 | "2분대가 2시를 맡는 줄 알았다." |
| **F7** | 스트레스 훈련 기록 | 지휘 위임 선언까지 95초 소요, 위임 문구가 정해져 있지 않았음 |

| 팩트체크 예: F1 진술(30초)과 F2 기록(110초)이 다르다. F3를 보면 관측자는 제때 송신하려 했으나 무전이 점유되어 있었다. 원인은 '관측자 태만'이 아니라 '보고 채널 우선순위 규칙 없음(절차)'으로 분류한다. |
| --- |

다. 작성 서식 1 - 원인 분석 매트릭스

| **문제** | **확인된 사실** | **근거(F번호·기록)** | **원인 분류
(절차/도구/환경/숙련)** | **근본 원인** | **개선안** |
| --- | --- | --- | --- | --- | --- |
| **보고 지연** |  |  |  |  |  |
| **사격 구역 중복** |  |  |  |  |  |
| **지휘 위임 지연** |  |  |  |  |  |
|  |  |  |  |  |  |
|  |  |  |  |  |  |

라. 비난 표현 바꾸기

| **피할 표현** | **바꾼 표현(예시)** |
| --- | --- |
| **관측자가 게을러서 늦었다** | 무전 점유 시 긴급 보고 우선 규칙이 없어 보고가 110초 지연되었다 |
| **지휘자가 헷갈렸다** | 구역 할당표에 2~3시 중복이 있었고 확인 절차가 없었다 |

마. 작성 서식 2 - 개선점 목록(5건 이상) 및 SOP 1차 수정안

| **번호** | **개선점** | **SOP 현행** | **SOP 수정** | **근거(매트릭스 행)** |
| --- | --- | --- | --- | --- |
| **1** |  |  |  |  |
| **2** |  |  |  |  |
| **3** |  |  |  |  |
| **4** |  |  |  |  |
| **5** |  |  |  |  |
| **6** |  |  |  |  |

| **점검 항목** | **확인** |
| --- | --- |
| 4단계 순서로 AAR을 진행했다 | ( ) |
| 진술과 기록이 다른 곳을 모두 표시했다 | ( ) |
| 모든 원인 항목에 근거(F번호·기록)를 연결했다 | ( ) |
| 개선점이 5건 이상이다 | ( ) |
| 회의록에 개인 비난 표현이 없다 | ( ) |
| 개선점이 SOP 수정안에 반영되었다 | ( ) |

**

